In [ ]:
import os

os.environ["PYTENSOR_FLAGS"] = "cxx="

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LinearRegression
import causalpy as cp

# 1. Load Data
ncaa_nba = pd.read_csv("data/ncaa_nba_data.csv")

# 2. Reshape to Long Format
# Assuming columns are player_id, num_years_college, college_4...college_1, nba_season_1...nba_season_5
# We melt the dataset so each row is a player-time period
value_vars = [
    "college_4",
    "college_3",
    "college_2",
    "college_1",
    "nba_season_1",
    "nba_season_2",
    "nba_season_3",
    "nba_season_4",
    "nba_season_5",
]

long_df = ncaa_nba.melt(
    id_vars=["athlete_id", "num_years_college"],
    value_vars=value_vars,
    var_name="period",
    value_name="impact",
).dropna(subset=["impact"])

# 3. Create standardized 'time' variable (1 to 9) and Treatment indicators
time_map = {k: i + 1 for i, k in enumerate(value_vars)}
long_df["time"] = long_df["period"].map(time_map)

# Group indicators for OLS
for g in [2, 3, 4]:
    long_df[f"g{g}"] = (long_df["num_years_college"] == g).astype(int)

# Create a binary treatment indicator for CausalPy (1 if the player is in the NBA, 0 if in college)
long_df["treated"] = (long_df["time"] >= 5).astype(int)

In [14]:
# OLS with player FEs, time FEs, and group x time interactions
# Baseline: 1-year players (g1 omitted), reference period: college_1 (t=4)
formula = "impact ~ C(time) + g2*C(time) + g3*C(time) + g4*C(time) + C(athlete_id)"
ols_model = smf.ols(formula, data=long_df).fit(
    cov_type="cluster", cov_kwds={"groups": long_df["athlete_id"]}
)

# Extract group x time interaction coefficients
coefs = []
for g in [2, 3, 4]:
    for t in range(1, 10):
        matching = [
            p
            for p in ols_model.params.index
            if f"g{g}" in p and f"[T.{t}]" in p and "C(time)" in p
        ]
        if matching:
            p = matching[0]
            coefs.append(
                {
                    "group": g,
                    "time": t,
                    "coef": ols_model.params[p],
                    "ci_low": ols_model.conf_int().loc[p, 0],
                    "ci_high": ols_model.conf_int().loc[p, 1],
                }
            )

coef_df = pd.DataFrame(coefs)
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 impact   R-squared:                       0.771
Model:                            OLS   Adj. R-squared:                  0.726
Method:                 Least Squares   F-statistic:                 2.268e+10
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:25:30   Log-Likelihood:                -11243.
No. Observations:                5080   AIC:                         2.416e+04
Df Residuals:                    4241   BIC:                         2.965e+04
Df Model:                         838                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

/Users/gauravlaw/Desktop/Understanding Data Science/SP26_IDS701_UDS_Project/venv/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 841, but rank is 31
  warnings.warn('covariance of constraints does not have full '


In [15]:
# 1. We must add a 'treatment_time' column for CausalPy's new API.
# Since all players in our aligned dataset enter the NBA at time = 5 (nba_season_1),
# the treatment time is 5 for everyone.
long_df["treatment_time"] = 5

# 2. Define the CausalPy model using the top-level API
cp_model = cp.StaggeredDifferenceInDifferences(
    long_df,
    formula="impact ~ 1 + C(athlete_id) + C(time)",
    time_variable_name="time",
    unit_variable_name="athlete_id",
    treated_variable_name="treated",
    treatment_time_variable_name="treatment_time",
    model=cp.pymc_models.LinearRegression(),
)

# 3. Fit the model
result = cp_model.fit()

# 4. Print statistical summary of the treatment effects across the periods
print(result.summary())

Initializing NUTS using jitter+adapt_diag...



You can find the C code in this temporary file: /var/folders/rp/gf132r1n5bj14p__kxdf2hrm0000gp/T/pytensor_compilation_error_2cumt2cr


CompileError: Compilation failed (return status=1):
/usr/bin/clang++ -dynamiclib -g -O3 -fno-math-errno -Wno-unused-label -Wno-unused-variable -Wno-write-strings -Wno-c++11-narrowing -fno-exceptions -fno-unwind-tables -fno-asynchronous-unwind-tables -DNPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION -fPIC -undefined dynamic_lookup -ld64 -I/Users/gauravlaw/Desktop/Understanding Data Science/SP26_IDS701_UDS_Project/venv/lib/python3.14/site-packages/numpy/_core/include -I/opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14/include/python3.14 -I/Users/gauravlaw/Desktop/Understanding Data Science/SP26_IDS701_UDS_Project/venv/lib/python3.14/site-packages/pytensor/link/c/c_code -L/opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14/lib -fvisibility=hidden -o /Users/gauravlaw/.pytensor/compiledir_macOS-26.2-arm64-arm-64bit-Mach-O-arm-3.14.3-64/tmpk_aviwrg/m256099464c16b7a16c6fc8e8a9ef54e019c62357d98a4d6e1214d042eea4a7b8.so /Users/gauravlaw/.pytensor/compiledir_ma

In [16]:
# Plot the results using CausalPy's built-in plotting
fig, ax = result.plot()

# Customize plot aesthetics
fig.set_size_inches(12, 8)
ax.set_title(
    "Staggered DiD Event Study — NBA Impact by Extra College Years",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Time (Periods relative to NBA Draft)", fontsize=12)
ax.set_ylabel("Treatment Effect on Impact Score", fontsize=12)
ax.axvline(x=4.5, color="red", linestyle="--", alpha=0.5, label="Draft cutoff")
plt.legend()
plt.show()

NameError: name 'result' is not defined